# SkinIQ — EfficientNet-V2 training (proposal 3.6.1)

Trains two classifiers on a free Colab GPU and evaluates them on the held-out test split:

| Model | Classes |
|---|---|
| `skin_type` | combination, dry, normal, oily |
| `skin_condition` | acne, blackheads, enlarged_pores, hyperpigmentation, redness, wrinkles |

**Before you start**
1. **Runtime → Change runtime type → T4 GPU**.
2. Put the two dataset zips in a Google Drive folder (default: `My Drive/SkinIQ/`):
   `Skin Type Dataset - olimov.zip` and `Skin Issues Dataset - drishya.zip`.
3. **Runtime → Run all**. Results are saved to `My Drive/SkinIQ/models/` after every epoch, so a disconnect doesn't lose the best model.

Expected time on a T4: roughly 20–30 min for skin type and 40–60 min for skin condition.

## 1. Settings

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/SkinIQ"        # folder holding the two zips
SKIN_TYPE_ZIP = "Skin Type Dataset - olimov.zip"
SKIN_CONDITION_ZIP = "Skin Issues Dataset - drishya.zip"

EPOCHS = 20          # early stopping usually ends sooner
BATCH_SIZE = 32      # lower to 16 if you get "CUDA out of memory"
TASKS = ["skin_type", "skin_condition"]

## 2. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again"
print(torch.cuda.get_device_name(0))

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
for z in (SKIN_TYPE_ZIP, SKIN_CONDITION_ZIP):
    path = os.path.join(DRIVE_DIR, z)
    assert os.path.exists(path), f"Not found: {path} — check DRIVE_DIR and the zip names in step 1"
print("Both zips found in", DRIVE_DIR)

## 4. Get the SkinIQ code

In [ ]:
%cd /content
!rm -rf SkinIQ && git clone -q https://github.com/rwiie/SkinIQ.git
%cd /content/SkinIQ
!git log --oneline -1
!pip install -q "timm>=1.0"

## 5. Unzip and prepare the data (proposal 3.5)

Unzips to the Colab disk (much faster than reading from Drive) and runs the same
`prepare_data.py` as the repo, so the 70/15/15 split is identical to `training/data_report.md`.

In [ ]:
!mkdir -p data/raw
!cp "{DRIVE_DIR}/{SKIN_TYPE_ZIP}" "{DRIVE_DIR}/{SKIN_CONDITION_ZIP}" /tmp/
!unzip -q -o "/tmp/{SKIN_TYPE_ZIP}" -x "__MACOSX/*" -d data/raw
!unzip -q -o "/tmp/{SKIN_CONDITION_ZIP}" -x "__MACOSX/*" -d data/raw
!rm -rf data/raw/skin_type data/raw/skin_condition
!mv "data/raw/Skin Type Dataset - olimov" data/raw/skin_type
!mv "data/raw/Skin Issues Dataset - drishya" data/raw/skin_condition
!python training/prepare_data.py
!git diff --stat training/data_report.md || true

The last line should print nothing: an empty diff means the split matches the committed report.

## 6. Train and evaluate

Each run saves the best epoch (by validation macro-F1) and then reports accuracy, precision,
recall and F1 on the test split.

In [ ]:
OUT_DIR = f"{DRIVE_DIR}/models"
for task in TASKS:
    !python training/train.py --task {task} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --out-dir "{OUT_DIR}"

## 7. Results

In [ ]:
import json, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

for task in TASKS:
    d = f"{DRIVE_DIR}/models/{task}"
    m = json.load(open(f"{d}/metrics.json"))
    print(f"=== {task}: best epoch {m['best_epoch']} of {m['epochs_run']}")
    print(f"Test accuracy {m['test_accuracy']:.2%} | macro precision {m['test_macro']['precision']:.2%} "
          f"| macro recall {m['test_macro']['recall']:.2%} | macro F1 {m['test_macro']['f1-score']:.2%}")
    display(pd.DataFrame(m["per_class"]).T.round(3))

    h = pd.read_csv(f"{d}/history.csv")
    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.5))
    h.plot(x="epoch", y=["train_loss", "val_loss"], ax=a, title=f"{task} — loss")
    h.plot(x="epoch", y=["val_accuracy", "val_macro_f1"], ax=b, title=f"{task} — validation")
    fig.tight_layout(); fig.savefig(f"{d}/training_curves.png", dpi=150); plt.show()
    display(Image(f"{d}/confusion_matrix.png", width=520))

## 8. Done

Your Drive now has `SkinIQ/models/skin_type/` and `SkinIQ/models/skin_condition/`, each with
`model.pt`, `metrics.json`, `history.csv`, `confusion_matrix.png` and `training_curves.png`.

Share the `models` folder (Anyone with the link → Viewer) so the models can be added to the app in the next steps.